In [17]:
!git clone https://github.com/AlexTanui/HydrogenProductionMaterials_Discovery.git
%cd HydrogenProductionMaterials_Discovery

fatal: destination path 'HydrogenProductionMaterials_Discovery' already exists and is not an empty directory.
/content/HydrogenProductionMaterials_Discovery


In [18]:
!pip install torch-geometric pyyaml --quiet

In [19]:
%%writefile ml/config.py
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import yaml


@dataclass
class DataConfig:
    dataset: str = "md17"
    molecule: str = "ethanol"
    theory: str = "dft"
    gold_path: str = ""
    cutoff_radius: float = 5.0
    train_split: float = 0.8
    val_split: float = 0.1
    test_split: float = 0.1
    use_literature_split: bool = False
    seed: int = 42

    def __post_init__(self) -> None:
        total = self.train_split + self.val_split + self.test_split
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"train/val/test splits must sum to 1.0, got {total}")


@dataclass
class ModelConfig:
    phase: str = "mpnn"
    hidden_channels: int = 128
    num_layers: int = 4
    num_rbf: int = 50
    num_mc_samples: int = 20
    perturbation_strength: float = 0.1

    def __post_init__(self) -> None:
        valid_phases = ("mpnn", "blip", "graph_stochastic")
        if self.phase not in valid_phases:
            raise ValueError(f"model.phase must be one of {valid_phases}, got {self.phase!r}")


@dataclass
class TrainConfig:
    epochs: int = 200
    batch_size: int = 32
    lr: float = 5e-4
    energy_loss_weight: float = 1.0
    force_loss_weight: float = 100.0
    patience: int = 20
    device: str = "cpu"


@dataclass
class ExperimentConfig:
    name: str = "unnamed_experiment"
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)

    @classmethod
    def from_yaml(cls, path: str | Path) -> "ExperimentConfig":
        path = Path(path)
        with open(path) as f:
            raw = yaml.safe_load(f)

        return cls(
            name=raw.get("name", path.stem),
            data=DataConfig(**raw.get("data", {})),
            model=ModelConfig(**raw.get("model", {})),
            train=TrainConfig(**raw.get("train", {})),
        )

    def checkpoint_path(self, checkpoints_dir: str | Path = "experiments/checkpoints") -> Path:
        return Path(checkpoints_dir) / f"{self.name}.pt"

    def results_path(self, results_dir: str | Path = "experiments/results") -> Path:
        return Path(results_dir) / f"{self.name}.json"

Overwriting ml/config.py


In [20]:
%%writefile ml/models/mpnn.py
from __future__ import annotations

from typing import Optional

import torch
import torch.nn as nn

MAX_ATOMIC_NUMBER = 100


class GaussianRBF(nn.Module):

    def __init__(self, num_rbf: int = 50, cutoff: float = 5.0):
        super().__init__()
        centers = torch.linspace(0.0, cutoff, num_rbf)
        width = centers[1] - centers[0] if num_rbf > 1 else torch.tensor(cutoff)
        self.register_buffer("centers", centers)
        self.width = width

    def forward(self, distances: torch.Tensor) -> torch.Tensor:
        d = distances.view(-1, 1)
        return torch.exp(-((d - self.centers.view(1, -1)) ** 2) / (2 * self.width ** 2))


class MPNNLayer(nn.Module):

    def __init__(self, hidden_channels: int, num_rbf: int):
        super().__init__()
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_channels + num_rbf, hidden_channels),
            nn.SiLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )
        self.update_mlp = nn.Sequential(
            nn.Linear(2 * hidden_channels, hidden_channels),
            nn.SiLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )

    def forward(self, h: torch.Tensor, edge_index: torch.Tensor, edge_rbf: torch.Tensor) -> torch.Tensor:
        row, col = edge_index
        messages = self.message_mlp(torch.cat([h[col], edge_rbf], dim=-1))

        aggregated = torch.zeros_like(h)
        aggregated.index_add_(0, row, messages)

        return h + self.update_mlp(torch.cat([h, aggregated], dim=-1))


class MPNN(nn.Module):
    def __init__(self, hidden_channels: int = 128, num_layers: int = 4,
                 num_rbf: int = 50, cutoff: float = 5.0):
        super().__init__()
        self.embedding = nn.Embedding(MAX_ATOMIC_NUMBER + 1, hidden_channels)
        self.rbf = GaussianRBF(num_rbf=num_rbf, cutoff=cutoff)
        self.layers = nn.ModuleList([MPNNLayer(hidden_channels, num_rbf) for _ in range(num_layers)])
        self.readout = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.SiLU(),
            nn.Linear(hidden_channels, 1),
        )

    def predict_energy(self, z: torch.Tensor, edge_index: torch.Tensor,
                        edge_attr: torch.Tensor, batch: Optional[torch.Tensor] = None) -> torch.Tensor:
        h = self.embedding(z.view(-1))
        edge_rbf = self.rbf(edge_attr.view(-1))

        for layer in self.layers:
            h = layer(h, edge_index, edge_rbf)

        atomic_energies = self.readout(h).view(-1)

        if batch is None:
            return atomic_energies.sum().view(1)

        n_graphs = int(batch.max().item()) + 1
        total_energy = torch.zeros(n_graphs, dtype=atomic_energies.dtype, device=atomic_energies.device)
        total_energy.index_add_(0, batch, atomic_energies)
        return total_energy

    def predict_energy_and_forces(self, z: torch.Tensor, pos: torch.Tensor,
                                   edge_index: torch.Tensor, edge_attr: torch.Tensor,
                                   batch: Optional[torch.Tensor] = None) -> tuple[torch.Tensor, torch.Tensor]:
        pos = pos.requires_grad_(True)
        energy = self.predict_energy(z, edge_index, edge_attr, batch=batch)

        grad_outputs = torch.ones_like(energy)
        (dE_dR,) = torch.autograd.grad(
            outputs=energy, inputs=pos, grad_outputs=grad_outputs,
            create_graph=self.training, retain_graph=True,
        )
        forces = -dE_dR
        return energy, forces

Overwriting ml/models/mpnn.py


In [21]:
%%writefile experiments/configs/phase1_baseline.yaml
name: phase1_baseline

data:
  dataset: md17
  molecule: ethanol
  theory: dft
  gold_path: data/gold/md17_ethanol_dft
  cutoff_radius: 5.0
  train_split: 0.8
  val_split: 0.1
  test_split: 0.1
  use_literature_split: false
  seed: 42

model:
  phase: mpnn
  hidden_channels: 128
  num_layers: 4
  num_rbf: 50

train:
  epochs: 200
  batch_size: 32
  lr: 0.0005
  energy_loss_weight: 1.0
  force_loss_weight: 100.0
  patience: 20
  device: cpu

Overwriting experiments/configs/phase1_baseline.yaml


In [22]:
print("=== ml/data/datasets.py ===")
!cat ml/data/datasets.py

print("\n=== Does data/gold/ exist? ===")
import os
print(os.path.exists("data/gold"))

=== ml/data/datasets.py ===
"""PyTorch Geometric Dataset wrapper over gold-stage MD17 data.

Reads only from data/gold/ (see glossary.md SS3) - never bronze/silver
directly. Migrated from rajcleaning.ipynb; see git history for changes.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
from torch_geometric.data import Data, Dataset

from ml.data.md17 import build_graph


class MD17Dataset(Dataset):
    def __init__(self, gold_path: str | Path, split: str = "train", cutoff_radius: float = 5.0, num_rbf: int = 16):
        super().__init__()
        if split not in ("train", "val", "test"):
            raise ValueError(f"split must be train/val/test, got {split!r}")
        data = np.load(gold_path, allow_pickle=True)
        self.molecule = str(data["molecule"])
        self.theory = str(data["theory"])
        self.cutoff_radius = cutoff_radius
        self.num_rbf = num_rbf
        self.split = split
        idx = data[f"{split}_idx"]
    

In [23]:
!python -m ml.training.train experiments/configs/phase1_baseline.yaml